##### Copyright 2025 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini ネイティブ画像生成完全ガイド（Nano-Banana Pro）

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Nanobanana_Pro_Guide_ja.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Geminiは会話形式で画像を生成・処理できます。[高速なGemini 2.5 Flash（別名Nano Banana）または高度なGemini 3 Pro Preview（別名Nano Banana Pro）](https://ai.google.dev/gemini-api/docs/image-generation#model-selection)の画像モデルに、テキスト、画像、またはその両方でプロンプトを送ることで、これまでにない制御性でビジュアルを作成、編集、反復できます：

- **テキスト、画像、複数画像から画像へ：** テキスト記述から高品質画像を生成、テキストプロンプトで画像を編集・調整、複数入力画像で新シーンを構成やスタイル転送
- **反復的改善：** 会話形式で複数ターンにわたり画像を洗練、完璧になるまで微調整
- **高精度テキストレンダリング：** ロゴ、図、ポスターに最適な、読みやすく適切配置されたテキストを含む画像を正確に生成

生成画像すべてに[SynthID透かし](https://ai.google.dev/responsible/docs/safeguards/synthid)が含まれます。

## 画像生成（テキストから画像へ）

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

prompt = (
    "Create a picture of a nano banana dish in a fancy restaurant with a Gemini theme"
)

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[prompt],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("generated_image.png")

![AI生成 - ナノバナナ料理](https://ai.google.dev/static/gemini-api/docs/images/nano-banana.png)

Geminiテーマレストランのナノバナナ料理

## 画像編集（テキストと画像から画像へ）

**注意：** アップロードする画像の必要権利があることを確認してください。他者の権利侵害コンテンツ（欺瞞、嫌がらせ、危害の動画・画像含む）を生成しないでください。この生成AIサービス使用は[禁止使用ポリシー](https://policies.google.com/terms/generative-ai/use-policy)の対象です。

画像を提供しテキストプロンプトで要素追加・削除・変更、スタイル変更、カラーグレーディング調整ができます。

以下の例はbase64エンコード画像アップロードを示します。複数画像、大ペイロード、サポートMIMEタイプは[画像理解](https://ai.google.dev/gemini-api/docs/image-understanding)ページ参照。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

prompt = (
    "Create a picture of my cat eating a nano-banana in a "
    "fancy restaurant under the Gemini constellation",
)

image = Image.open("/path/to/cat_image.png")

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[prompt, image],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("generated_image.png")

![AI生成 - 猫がナノバナナ](https://ai.google.dev/static/gemini-api/docs/images/cat-banana.png)

猫がナノバナナを食べるAI生成画像

### マルチターン画像編集

会話形式で画像生成・編集を継続できます。チャットまたはマルチターン会話は画像反復改善の推奨方法です。以下は光合成インフォグラフィック生成プロンプトの例です。

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()

chat = client.chats.create(
    model="gemini-3-pro-image-preview",
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
        tools=[{"google_search": {}}]
    )
)

message = "Create a vibrant infographic that explains photosynthesis as if it were a recipe for a plant's favorite food. Show the \"ingredients\" (sunlight, water, CO2) and the \"finished dish\" (sugar/energy). The style should be like a page from a colorful kids' cookbook, suitable for a 4th grader."

response = chat.send_message(message)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("photosynthesis.png")

![AI生成 - 光合成インフォグラフィック](https://ai.google.dev/static/gemini-api/docs/images/infographic-eng.png)

光合成のAI生成インフォグラフィック

同じチャットでグラフィック言語をスペイン語に変更できます。

In [ ]:
message = "Update this infographic to be in Spanish. Do not change any other elements of the image."
aspect_ratio = "16:9" # "1:1","2:3","3:2","3:4","4:3","4:5","5:4","9:16","16:9","21:9"
resolution = "2K" # "1K", "2K", "4K"

response = chat.send_message(message,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution
        ),
    ))

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("photosynthesis_spanish.png")

![AI生成 - 光合成（スペイン語）](https://ai.google.dev/static/gemini-api/docs/images/infographic-spanish.png)

スペイン語版光合成インフォグラフィック

## Gemini 3 Pro Image の新機能

Gemini 3 Pro Image（`gemini-3-pro-image-preview`）は、プロアセット制作最適化の最先端画像生成・編集モデルです。高度推論で最困難ワークフロー対応設計、複雑マルチターン作成・変更タスクに優れます。

- **高解像度出力：** 1K、2K、4Kビジュアル生成機能内蔵
- **高度テキストレンダリング：** インフォグラフィック、メニュー、図、マーケティング資産用の読みやすくスタイライズされたテキスト生成可能
- **Google検索グラウンディング：** モデルはGoogle検索ツール使用で事実確認、リアルタイムデータ（天気図、株価チャート、最近イベント）基準画像生成
- **思考モード：** 複雑プロンプト推論に「思考」プロセス利用。最終高品質出力前に中間「思考画像」（バックエンド表示、課金なし）で構成改善
- **最大14参照画像：** 最終画像生成に最大14参照画像ミックス可能

### 最大14参照画像使用

Gemini 3 Pro Previewは最大14参照画像ミックス可能。14画像には：

- 最終画像高精度含有対象物画像（最大6）
- キャラクター一貫性維持人間画像（最大5）

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

prompt = "An office group photo of these people, they are making funny faces."
aspect_ratio = "5:4" # "1:1","2:3","3:2","3:4","4:3","4:5","5:4","9:16","16:9","21:9"
resolution = "2K" # "1K", "2K", "4K"

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=[
        prompt,
        Image.open('person1.png'),
        Image.open('person2.png'),
        Image.open('person3.png'),
        Image.open('person4.png'),
        Image.open('person5.png'),
    ],
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution
        ),
    )
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("office.png")

![AI生成 - オフィス集合写真](https://ai.google.dev/static/gemini-api/docs/images/office-group-photo.jpeg)

AI生成オフィス集合写真

### Google検索グラウンディング

[Google検索ツール](https://ai.google.dev/gemini-api/docs/google-search)で天気予報、株価チャート、最近イベント等リアルタイム情報基準画像生成。

注意：画像生成でGoogle検索グラウンディング使用時、画像検索結果は生成モデル未送信、レスポンス除外。

In [ ]:
from google import genai
prompt = "Visualize the current weather forecast for the next 5 days in San Francisco as a clean, modern weather chart. Add a visual on what I should wear each day"
aspect_ratio = "16:9" # "1:1","2:3","3:2","3:4","4:3","4:5","5:4","9:16","16:9","21:9"

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        ),
        tools=[{"google_search": {}}]
    )
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("weather.png")

### 最大4K解像度画像生成

Gemini 3 Pro Imageは1K、2K、4K画像生成可能。1Kと2Kは同料金なので2K自由使用可。4Kは高価なため必要時のみ使用（[料金](https://ai.google.dev/gemini-api/docs/pricing#gemini-2.5-flash-image)参照）。

In [ ]:
from google import genai
from google.genai import types

prompt = "Da Vinci style anatomical sketch of a dissected Monarch butterfly. Detailed drawings of the head, wings, and legs on textured parchment with notes in English." 
aspect_ratio = "1:1" # "1:1","2:3","3:2","3:4","4:3","4:5","5:4","9:16","16:9","21:9"
resolution = "1K" # "1K", "2K", "4K"

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['TEXT', 'IMAGE'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution
        ),
    )
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("butterfly.png")

### 思考プロセス

Gemini 3 Pro Imageは思考モデルのため、最終出力前にモデルが辿った思考プロセス確認可能。

In [ ]:
for part in response.parts:
    if part.thought:
        if part.text:
            print(part.text)
        elif image:= part.as_image():
            image.show()

## その他画像生成モード

Gemini APIは[Imagen](https://ai.google.dev/gemini-api/docs/imagen)経由で追加画像生成機能提供。詳細は[Imagenドキュメント](https://ai.google.dev/gemini-api/docs/imagen)参照。

## バッチ画像生成

[Batch API](https://ai.google.dev/gemini-api/docs/batch)で複数画像生成リクエスト非同期処理可能。大規模画像生成タスクコスト削減。

## プロンプトガイドと戦略

画像生成を習得するには、次の基本原則から始めます：
> **シーンを説明し、キーワードを単に列挙しないでください。** モデルの中核的な強みは深い言語理解力です。物語的で説明的な段落は、ほとんどの場合、切断されたキーワードのリストよりも優れた、より一貫性のある画像を生成します。

### 画像生成用プロンプト

以下の戦略は、探している画像を正確に生成するための効果的なプロンプト作成に役立ちます。

#### 1. フォトリアリスティックシーン

リアルな画像の場合は、写真用語を使用します。カメラアングル、レンズタイプ、照明、細部を記載してモデルをフォトリアリスティックな結果に導きます。

**テンプレート：**
```
A photorealistic [shot type] of [subject], [action or expression], set in
[environment]. The scene is illuminated by [lighting description], creating
a [mood] atmosphere. Captured with a [camera/lens details], emphasizing
[key textures and details]. The image should be in a [aspect ratio] format.
```

**プロンプト例：**
```
A photorealistic close-up portrait of an elderly Japanese ceramicist with
deep, sun-etched wrinkles and a warm, knowing smile. He is carefully
inspecting a freshly glazed tea bowl. The setting is his rustic,
sun-drenched workshop. The scene is illuminated by soft, golden hour light
streaming through a window, highlighting the fine texture of the clay.
Captured with an 85mm portrait lens, resulting in a soft, blurred background
(bokeh). The overall mood is serene and masterful. Vertical portrait
orientation.
```

In [ ]:
from google import genai
from google.genai import types    

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A photorealistic close-up portrait of an elderly Japanese ceramicist with deep, sun-etched wrinkles and a warm, knowing smile. He is carefully inspecting a freshly glazed tea bowl. The setting is his rustic, sun-drenched workshop with pottery wheels and shelves of clay pots in the background. The scene is illuminated by soft, golden hour light streaming through a window, highlighting the fine texture of the clay and the fabric of his apron. Captured with an 85mm portrait lens, resulting in a soft, blurred background (bokeh). The overall mood is serene and masterful.",
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("photorealistic_example.png")

![フォトリアリスティック例](https://ai.google.dev/static/gemini-api/docs/images/photorealistic_example.png)

年配の日本人陶芸家のフォトリアリスティッククローズアップポートレート

#### 2. スタイライズドイラストとステッカー

ステッカー、アイコン、アセットを作成するには、スタイルを明示的に指定し、透明な背景をリクエストします。

**テンプレート：**
```
A [style] sticker of a [subject], featuring [key characteristics] and a
[color palette]. The design should have [line style] and [shading style].
The background must be transparent.
```

**プロンプト例：**
```
A kawaii-style sticker of a happy red panda wearing a tiny bamboo hat. It's
munching on a green bamboo leaf. The design features bold, clean outlines,
simple cel-shading, and a vibrant color palette. The background must be white.
```

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A kawaii-style sticker of a happy red panda wearing a tiny bamboo hat. It's munching on a green bamboo leaf. The design features bold, clean outlines, simple cel-shading, and a vibrant color palette. The background must be white.",
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("red_panda_sticker.png")

![レッドパンダステッカー](https://ai.google.dev/static/gemini-api/docs/images/red_panda_sticker.png)

かわいいレッドパンダステッカー

#### 3. 画像内の正確なテキスト

Geminiはテキストレンダリングに優れています。テキスト、フォントスタイル（記述的）、全体的なデザインについて明確にしてください。プロフェッショナルアセット制作にはGemini 3 Pro Image Previewを使用してください。

**テンプレート：**
```
Create a [image type] for [brand/concept] with the text "[text to render]"
in a [font style]. The design should be [style description], with a
[color scheme].
```

**プロンプト例：**
```
Create a modern, minimalist logo for a coffee shop called 'The Daily Grind'.
The text should be in a clean, bold, sans-serif font. The color scheme is
black and white. Put the logo in a circle. Use a coffee bean in a clever way.
```

In [ ]:
from google import genai
from google.genai import types    

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents="Create a modern, minimalist logo for a coffee shop called 'The Daily Grind'. The text should be in a clean, bold, sans-serif font. The color scheme is black and white. Put the logo in a circle. Use a coffee bean in a clever way.",
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio="1:1",
        )
    )
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("logo_example.jpg")

![ロゴ例](https://ai.google.dev/static/gemini-api/docs/images/logo_example.jpg)

モダンでミニマリストなコーヒーショップロゴ

#### 4. 商品モックアップとコマーシャルフォトグラフィー

eコマース、広告、ブランディング用のクリーンでプロフェッショナルな商品写真の作成に最適です。

**テンプレート：**
```
A high-resolution, studio-lit product photograph of a [product description]
on a [background surface/description]. The lighting is a [lighting setup,
e.g., three-point softbox setup] to [lighting purpose]. The camera angle is
a [angle type] to showcase [specific feature]. Ultra-realistic, with sharp
focus on [key detail]. [Aspect ratio].
```

**プロンプト例：**
```
A high-resolution, studio-lit product photograph of a minimalist ceramic
coffee mug in matte black, presented on a polished concrete surface. The
lighting is a three-point softbox setup designed to create soft, diffused
highlights and eliminate harsh shadows. The camera angle is a slightly
elevated 45-degree shot to showcase its clean lines. Ultra-realistic, with
sharp focus on the steam rising from the coffee. Square image.
```

In [ ]:
from google import genai
from google.genai import types

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A high-resolution, studio-lit product photograph of a minimalist ceramic coffee mug in matte black, presented on a polished concrete surface. The lighting is a three-point softbox setup designed to create soft, diffused highlights and eliminate harsh shadows. The camera angle is a slightly elevated 45-degree shot to showcase its clean lines. Ultra-realistic, with sharp focus on the steam rising from the coffee. Square image.",
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("product_mockup.png")

![商品モックアップ](https://ai.google.dev/static/gemini-api/docs/images/product_mockup.png)

ミニマリストセラミックマグの高解像度商品写真

#### 5. ミニマリストとネガティブスペースデザイン

テキストをオーバーレイするウェブサイト、プレゼンテーション、マーケティング資料の背景作成に最適です。

**テンプレート：**
```
A minimalist composition featuring a single [subject] positioned in the
[bottom-right/top-left/etc.] of the frame. The background is a vast, empty
[color] canvas, creating significant negative space. Soft, subtle lighting.
[Aspect ratio].
```

**プロンプト例：**
```
A minimalist composition featuring a single, delicate red maple leaf
positioned in the bottom-right of the frame. The background is a vast, empty
off-white canvas, creating significant negative space for text. Soft,
diffused lighting from the top left. Square image.
```

In [ ]:
from google import genai
from google.genai import types    

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents="A minimalist composition featuring a single, delicate red maple leaf positioned in the bottom-right of the frame. The background is a vast, empty off-white canvas, creating significant negative space for text. Soft, diffused lighting from the top left. Square image.",
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("minimalist_design.png")

#### 6. 連続アート（コミックパネル/ストーリーボード）

コミックパネルやストーリーボードシーケンスの作成に使用できます。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

image_input = Image.open('/path/to/your/man_in_white_glasses.jpg')
text_input = "Make a 3 panel comic in a gritty, noir art style with high-contrast black and white inks. Put the character in a humurous scene."

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=[text_input, image_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("comic_panel.jpg")

#### 7. Google検索グラウンディング使用

リアルタイム情報に基づく画像生成にGoogle検索を活用します。

In [ ]:
from google import genai
from google.genai import types
prompt = "Make a simple but stylish graphic of last night's Arsenal game in the Champion's League"
aspect_ratio = "16:9" # "1:1","2:3","3:2","3:4","4:3","4:5","5:4","9:16","16:9","21:9"

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Text', 'Image'],
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
        ),
        tools=[{"google_search": {}}]
    )
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif image:= part.as_image():
        image.save("football-score.jpg")

### 画像編集用プロンプト

以下の戦略は、画像を正確に編集するための効果的なプロンプト作成に役立ちます。

#### 1. 要素の追加と削除

画像から要素を追加または削除するには、変更内容を明確に説明してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

# Base image prompt: "A photorealistic picture of a fluffy ginger cat sitting on a wooden floor, looking directly at the camera. Soft, natural light from a window."
image_input = Image.open('/path/to/your/cat_photo.png')
text_input = """Using the provided image of my cat, please add a small, knitted wizard hat on its head. Make it look like it's sitting comfortably and not falling off."""

# Generate an image from a text prompt
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[text_input, image_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("cat_with_hat.png")

#### 2. インペインティング（セマンティックマスキング）

画像の特定領域を変更するには、変更する領域とその変更内容を説明してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

# Base image prompt: "A wide shot of a modern, well-lit living room with a prominent blue sofa in the center. A coffee table is in front of it and a large window is in the background."
living_room_image = Image.open('/path/to/your/living_room.png')
text_input = """Using the provided image of a living room, change only the blue sofa to be a vintage, brown leather chesterfield sofa. Keep the rest of the room, including the pillows on the sofa and the lighting, unchanged."""

# Generate an image from a text prompt
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[living_room_image, text_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("living_room_edited.png")

#### 3. スタイル転送

画像のスタイルを変更するには、希望するスタイルを説明してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

# Base image prompt: "A photorealistic, high-resolution photograph of a busy city street in New York at night, with bright neon signs, yellow taxis, and tall skyscrapers."
city_image = Image.open('/path/to/your/city.png')
text_input = """Transform the provided photograph of a modern city street at night into the artistic style of Vincent van Gogh's 'Starry Night'. Preserve the original composition of buildings and cars, but render all elements with swirling, impasto brushstrokes and a dramatic palette of deep blues and bright yellows."""

# Generate an image from a text prompt
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[city_image, text_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("city_style_transfer.png")

#### 4. 高度な合成：複数画像の結合

複数の画像を結合するには、各画像の役割と最終的な構成を説明してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

# Base image prompts:
# 1. Dress: "A professionally shot photo of a blue floral summer dress on a plain white background, ghost mannequin style."
# 2. Model: "Full-body shot of a woman with her hair in a bun, smiling, standing against a neutral grey studio background."
dress_image = Image.open('/path/to/your/dress.png')
model_image = Image.open('/path/to/your/model.png')

text_input = """Create a professional e-commerce fashion photo. Take the blue floral dress from the first image and let the woman from the second image wear it. Generate a realistic, full-body shot of the woman wearing the dress, with the lighting and shadows adjusted to match the outdoor environment."""

# Generate an image from a text prompt
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[dress_image, model_image, text_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("fashion_ecommerce_shot.png")

#### 5. 高精度ディテール保持

重要なディテールを保持しながら編集するには、保持する要素を明示してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

# Base image prompts:
# 1. Woman: "A professional headshot of a woman with brown hair and blue eyes, wearing a plain black t-shirt, against a neutral studio background."
# 2. Logo: "A simple, modern logo with the letters 'G' and 'A' in a white circle."
woman_image = Image.open('/path/to/your/woman.png')
logo_image = Image.open('/path/to/your/logo.png')
text_input = """Take the first image of the woman with brown hair, blue eyes, and a neutral expression. Add the logo from the second image onto her black t-shirt. Ensure the woman's face and features remain completely unchanged. The logo should look like it's naturally printed on the fabric, following the folds of the shirt."""

# Generate an image from a text prompt
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[woman_image, logo_image, text_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("woman_with_logo.png")

#### 6. 命を吹き込む

静的な画像を動的に見せるには、アクションや動きを説明してください。

In [ ]:
from google import genai
from PIL import Image

client = genai.Client()

# Base image prompt: "A rough pencil sketch of a flat sports car on white paper."
sketch_image = Image.open('/path/to/your/car_sketch.png')
text_input = """Turn this rough pencil sketch of a futuristic car into a polished photo of the finished concept car in a showroom. Keep the sleek lines and low profile from the sketch but add metallic blue paint and neon rim lighting."""

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=[sketch_image, text_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("car_photo.png")

#### 7. キャラクター一貫性：360度ビュー

複数の角度からのキャラクター一貫性を維持するには、同じキャラクターであることを明示してください。

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

client = genai.Client()

image_input = Image.open('/path/to/your/man_in_white_glasses.jpg')
text_input = """A studio portrait of this man against white, in profile looking right"""

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=[text_input, image_input],
)

for part in response.parts:
    if part.text is not None:
        print(part.text)
    elif part.inline_data is not None:
        image = part.as_image()
        image.save("man_right_profile.png")

## ベストプラクティス

効果的な画像生成のためのベストプラクティス：

- 具体的で詳細なプロンプト使用
- マルチターン会話で反復改善
- 適切なアスペクト比と解像度選択
- 必要に応じてGoogle検索グラウンディング活用
- 複雑タスクにはGemini 3 Pro Image使用検討

## 制限事項

現在の制限事項：

- Gemini 2.5 Flash Image：最大3参照画像
- Gemini 3 Pro Image：最大14参照画像
- 生成画像にはSynthID透かし含有
- コンテンツポリシー適用

## オプション設定

画像生成のオプション設定をカスタマイズできます：アスペクト比、解像度、レスポンスモダリティなど。

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[prompt],
    config=types.GenerateContentConfig(
        response_modalities=['Image']
    )
)

## モデル選択

適切なモデル選択：

- **Gemini 2.5 Flash Image：** 高速、低コスト、ほとんどのユースケースに最適
- **Gemini 3 Pro Image：** 複雑タスク、高解像度必要時、Google検索グラウンディング使用時

詳細は[モデル選択ガイド](https://ai.google.dev/gemini-api/docs/image-generation#model-selection)参照。

## 次のステップ

さらに学ぶ：

- [画像生成ドキュメント](https://ai.google.dev/gemini-api/docs/image-generation)
- [プロンプトガイド](https://ai.google.dev/gemini-api/docs/image-generation#prompt-guide)
- [Imagen](https://ai.google.dev/gemini-api/docs/imagen)
- [Gemini APIクックブック](https://github.com/google-gemini/cookbook)